# First pass at gathering geospatial and health data in the Netherlands.

Centraal Bureau voor de Statistiek, CBS StatLine as Open Data, OData API

Data description:

Official national statistics API for automated extraction of StatLine tables in OData format, suitable for population denominators, mortaility, hospital admissions, and many other health system relvant series.

Direct Link:

https://www.cbs.nl/en-gb/our-services/open-data/statline-as-open-data/quick-start-guide

Access method:

OData API (JSON)

In [ ]:
%pip install pandas
%pip install requests

In [ ]:
import pandas as pd
import requests
from pathlib import Path

def cbs_odata_typed_dataset(table_id: str, top: int = 5) -> pd.DataFrame:
    '''Fetch top rows from a CBS StatLine table via OData TypedDataSet.'''
    url = f'https://opendata.cbs.nl/ODataApi/OData/{table_id}/TypedDataSet'
    r = requests.get(url, params={'$top': str(top)}, timeout=60)
    r.raise_for_status()
    payload = r.json()
    df = pd.DataFrame(payload.get('value', []))

    data_dir = 'data'
    output_path = str(Path(data_dir) / Path(table_id).name) + '_CBS.csv'

    df.to_csv(output_path)

    return df

df = cbs_odata_typed_dataset('7233ENG', top=5)
print(df.head())

Status: Works.

Source Name:

Centraal Bureau voor de Statistiek, Deaths, cause of death (extensive list), age and sex table.

Data Description:

Cause specific mortailty coutns by underlying cause (extensive ICD three digit list), age, group, and sex. Supports mortality profiling, burde analysis, and trend monitoring.

Direct Link:

https://www.cbs.nl/en-gb/figures/detail/7233ENG

Access method:

OData API (table id 7233ENG), plus StatLine download options

In [ ]:
import pandas as pd
import requests

def fetch_cbs_deaths_extensive(top: int = 10) -> pd.DataFrame:
    '''Fetch a sample of cause of death records from CBS table 7233ENG via OData.'''
    url = 'https://opendata.cbs.nl/ODataApi/OData/7233ENG/TypedDataSet'
    r = requests.get(url, params={'$top': str(top)}, timeout=60)
    r.raise_for_status()

    df = pd.DataFrame(r.json().get('value', []))

    data_dir = 'data'
    output_path = str(Path(data_dir).name) + '/7233ENG_CBS.csv'

    df.to_csv(output_path)

    return df

df = fetch_cbs_deaths_extensive(top=10)
print(df.head())

Status: Works

Age 10000 makes no sense. Requires further analysis.

Source Name:

Centraal Burea voor de Statistiek, Ziekenhuiopnamen en patienten, diagnosis classification VTV (table 84067NED)

Data description:

Hospital admissins, admitted persons, inpatient days, and average length of stay for residents registered in BRP, with diagnosis lcassfiications. Useful for hopstial utilisation and burden analysis.

Direct Link:

https://www.cbs.nl/nl-nl/cijfers/detail/84067ned

Access method:

OData API (table id 84067NED)

In [ ]:
import pandas as pd
import requests

def fetch_cbs_hospital_admissions_vtv(top: int = 10) -> pd.DataFrame:
    '''Fetch a sample of hospital admissions from CBS table 84067NED via OData.'''
    url = 'https://opendata.cbs.nl/ODataApi/OData/84067NED/TypedDataSet'
    r = requests.get(url, params={'$top': str(top)}, timeout=60)
    r.raise_for_status()
    df = pd.DataFrame(r.json().get('value', []))

    path = 'data/' + '84067NED_CBS.csv'

    df.to_csv(path)

    return df

df = fetch_cbs_hospital_admissions_vtv(top=10)
print(df.head())

Status: Works

Language in dutch. Requires translation.

Source Name:

Centraal Bureau voor de Statistiek, Ziekenhuisopnamen, ISHMT diagnosis classification, region (table 84521NED)

Data Description:

Hospital admissions with regional breakdown, enabling subnational utilisation comparisons and linkage to regional denominators. Useful for benchmarking and geographic inequality analyses.

Direct link:

https://www.cbs.nl/nl-nl/cijfers/detail/84521NED

Access method:

OData API (table id 84521NED)

In [ ]:
import pandas as pd
import requests

def fetch_cbs_hospital_admissions_region(top: int = 10) -> pd.DataFrame:
    '''Fetch a sample of regional hospital admissions from CBS table 84521NED via OData.'''
    url = 'https://opendata.cbs.nl/ODataApi/OData/84521NED/TypedDataSet'
    r = requests.get(url, params={'$top': str(top)}, timeout=60)
    r.raise_for_status()
    df = pd.DataFrame(r.json().get('value', []))

    output_path = 'data/' + '84521NED_CBS.csv'

    df.to_csv(output_path)

    return df

df = fetch_cbs_hospital_admissions_region(top=10)
print(df.head())

Source Name:

Nederlandse Zorgautoriteit, Zorgbeeld, Wachttijden medisch specialistische zorg, dataset and OpenAPI

Data Description

Mandatory reported waiting times for elective specialist care, including treatment, outpatient vsisit, and diagnostics by specialty, with regular updates and historical CSV archives. Suitable for access and capacity monitoring.

Direct Link:

https://zorgbeeld.nza.nl/rest-doc/openapi

Access method:

REST API (OpenAPI), plus CSV downloads

In [ ]:
import io
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def build_session() -> requests.Session:
    '''Create a requests session with browser-like headers and retry logic.'''
    session = requests.Session()

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=[403, 429, 500, 502, 503, 504],
        allowed_methods=['GET'],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)

    session.headers.update({
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/145.0.0.0 Safari/537.36'
        ),
        'Accept': 'text/csv,application/json,text/plain,*/*',
        'Accept-Language': 'en-US,en;q=0.9,nl;q=0.8',
        'Referer': 'https://puc.overheid.nl/nza/doc/PUC_651798_22/1/',
    })
    return session


def fetch_nza_waiting_times_csv() -> pd.DataFrame:
    '''Download the current NZa waiting-times CSV and return it as a dataframe.'''
    session = build_session()

    url = (
        'https://puc.overheid.nl/PUC/Handlers/DownloadDocument.ashx'
        '?identifier=PUC_809508_22&versienummer=1'
    )

    response = session.get(url, timeout=120)
    response.raise_for_status()

    content_type = response.headers.get('Content-Type', '').lower()
    if 'csv' not in content_type and 'text' not in content_type and 'octet-stream' not in content_type:
        raise ValueError(f'Unexpected content type: {content_type}')

    text = response.content.decode('utf-8-sig', errors='replace')
    df = pd.read_csv(io.StringIO(text), sep=None, engine='python')

    path = 'data/' + 'NZa_waiting_times.csv'

    df.to_csv(path)

    return df


df = fetch_nza_waiting_times_csv()
print(df.head())
print(df.shape)

Status: Works.

Source Name:

Nederlandse Zorgautoriteit, Open DIS data, medisch specialistische zorg (downloads)

Data descriptoin:

Aggregated diagnosis and care product activity derived from the DBC information system, including counts and average prices, updated monthly, and explicitly excluding patient and institution level microdata. Useful for service mix and national level case mix patterns.

Direct link:

https://opendisdata.nza.nl/

Access method:

CSV downloads (site downloads section)

In [ ]:
from __future__ import annotations

import io
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


OPEN_DIS_TABLES: dict[str, dict[str, str | bool]] = {
    'dbc': {
        'url': 'https://opendisdata.nza.nl/download/csv/01_DBC.csv',
        'kind': 'fact',
        'large': False,
    },
    'dbc_profiel': {
        'url': 'https://opendisdata.nza.nl/download/csv/02_DBC_PROFIEL.csv',
        'kind': 'fact',
        'large': True,
    },
    'ref_zorgproduct': {
        'url': 'https://opendisdata.nza.nl/download/csv/05_REF_ZPD.csv',
        'kind': 'reference',
        'large': False,
    },
    'ref_zorgactiviteit': {
        'url': 'https://opendisdata.nza.nl/download/csv/03_REF_ZAT.csv',
        'kind': 'reference',
        'large': False,
    },
    'ref_specialisme': {
        'url': 'https://opendisdata.nza.nl/download/csv/06_REF_SPC.csv',
        'kind': 'reference',
        'large': False,
    },
    'ref_diagnose': {
        'url': 'https://opendisdata.nza.nl/download/csv/04_REF_DGN.csv',
        'kind': 'reference',
        'large': False,
    },
    'top100_diagnose_json': {
        'url': 'https://opendisdata.nza.nl/download/diagnose/json/pat/100/',
        'kind': 'summary',
        'large': False,
    },
    'top100_zorgproduct_json': {
        'url': 'https://opendisdata.nza.nl/download/zorgproduct/json/pat/100/',
        'kind': 'summary',
        'large': False,
    },
}


def build_session() -> requests.Session:
    '''Create a resilient HTTP session with browser-like headers.'''
    session = requests.Session()

    retry = Retry(
        total=3,
        connect=3,
        read=3,
        backoff_factor=2.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=['GET', 'HEAD'],
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)

    session.headers.update({
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/145.0.0.0 Safari/537.36'
        ),
        'Accept': 'text/csv,application/json,application/octet-stream,text/plain,*/*',
        'Accept-Language': 'en-US,en;q=0.9,nl;q=0.8',
        'Referer': 'https://opendisdata.nza.nl/',
        'Connection': 'keep-alive',
    })
    return session


def download_small_csv(url: str) -> pd.DataFrame:
    '''Download a small CSV file directly into a dataframe.'''
    session = build_session()
    response = session.get(url, timeout=(30, 120), allow_redirects=True)
    response.raise_for_status()
    text = response.content.decode('utf-8-sig', errors='replace')
    return pd.read_csv(io.StringIO(text), low_memory=False)


def download_json(url: str) -> pd.DataFrame:
    '''Download a JSON endpoint and normalize it into a dataframe.'''
    session = build_session()
    response = session.get(url, timeout=(30, 120), allow_redirects=True)
    response.raise_for_status()
    payload = response.json()

    if isinstance(payload, list):
        return pd.DataFrame(payload)

    return pd.json_normalize(payload)


def get_remote_size(session: requests.Session, url: str) -> int | None:
    '''Attempt to read the remote Content-Length.'''
    try:
        response = session.head(url, timeout=(30, 60), allow_redirects=True)
        response.raise_for_status()
        value = response.headers.get('Content-Length')
        return int(value) if value is not None else None
    except Exception:
        return None


def download_with_resume(url: str, output_path: str, chunk_size: int = 4 * 1024 * 1024) -> Path:
    '''Download a large file with resume support using HTTP range requests.'''
    session = build_session()
    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    remote_size = get_remote_size(session, url)
    attempts = 0

    while True:
        attempts += 1
        if attempts > 20:
            raise RuntimeError(f'Exceeded maximum attempts while downloading {url}')

        local_size = path.stat().st_size if path.exists() else 0
        if remote_size is not None and local_size >= remote_size:
            return path

        headers: dict[str, str] = {}
        mode = 'wb'

        if local_size > 0:
            headers['Range'] = f'bytes={local_size}-'
            mode = 'ab'

        try:
            with session.get(
                url,
                headers=headers,
                stream=True,
                timeout=(30, 300),
                allow_redirects=True,
            ) as response:
                if response.status_code not in [200, 206]:
                    response.raise_for_status()

                with path.open(mode) as f:
                    for chunk in response.iter_content(chunk_size=chunk_size):
                        if chunk:
                            f.write(chunk)
                            f.flush()
        except requests.exceptions.RequestException:
            pass

        new_local_size = path.stat().st_size if path.exists() else 0

        if remote_size is not None:
            if new_local_size >= remote_size:
                return path
            if new_local_size <= local_size:
                raise RuntimeError(
                    f'Download stalled at {new_local_size} bytes out of {remote_size} bytes.'
                )
        else:
            if new_local_size > local_size:
                continue
            raise RuntimeError('Download interrupted and remote size could not be determined.')


def fetch_open_dis_table(name: str, data_dir: str = 'data') -> pd.DataFrame | Path:
    '''Fetch one Open DIS table either as a dataframe or as a local CSV path for large files.'''
    if name not in OPEN_DIS_TABLES:
        raise KeyError(f'Unknown table name: {name}')

    meta = OPEN_DIS_TABLES[name]
    url = str(meta['url'])
    is_large = bool(meta['large'])

    if url.endswith('.csv') and not is_large:
        return download_small_csv(url)

    if '/json/' in url:
        return download_json(url)

    output_path = str(Path(data_dir) / Path(url).name)
    return download_with_resume(url, output_path)


ref_spc = fetch_open_dis_table('ref_specialisme')
ref_dgn = fetch_open_dis_table('ref_diagnose')
ref_zpd = fetch_open_dis_table('ref_zorgproduct')
ref_zat = fetch_open_dis_table('ref_zorgactiviteit')

print(ref_spc.head())
print(ref_dgn.head())
print(ref_zpd.head())
print(ref_zat.head())

In [ ]:
dbc = fetch_open_dis_table('dbc')

output_path = 'data/' + 'NZa_dbc.csv'

dbc.to_csv(output_path)

dbc_profiel = fetch_open_dis_table('dbc_profiel')

print(dbc.head())
#print(dbc_profiel.head()) #dbc_profiel is downloaded to ../data path.

Status: Working after refining the downloader to stream and resume the download and download a limited amount of bytes at once.

Source name:

Rijksinstituut voor Volksgezondheid en Milieu, Regiobeeld, Open dataa

Data description

Regional indicator platform for care, health, and wellbeing, including acute care indicators at ROAZ level, with download functionality per figure. Useful for subnational benchmarking and integrated regional profiles.

Direct Link:

https://www.regiobeeld.nl/open-data

Access method:

Dashboard export (per chart or map downloads, multiple formats)

In [ ]:
def inspect_rivm_statline_response(table_id: str, top: int = 5) -> None:
    '''Print diagnostic information about a RIVM StatLine response.'''
    session = build_session()
    url = f'https://statline.rivm.nl/ODataApi/OData/{table_id}/TypedDataSet'
    params = {
        '$format': 'json',
        '$top': str(top),
    }

    response = session.get(url, params=params, timeout=120)
    response.raise_for_status()

    print('Final URL:', response.url)
    print('Status:', response.status_code)
    print('Content-Type:', response.headers.get('Content-Type'))
    print('First 500 characters:')
    print(response.text[:500])


inspect_rivm_statline_response('50148NED', top=5)

In [ ]:
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def build_session() -> requests.Session:
    '''Create a requests session with browser-like headers and retry logic.'''
    session = requests.Session()

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=['GET'],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)

    session.headers.update({
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/145.0.0.0 Safari/537.36'
        ),
        'Accept': 'application/json, application/xml, text/plain, */*',
        'Accept-Language': 'en-US,en;q=0.9,nl;q=0.8',
    })
    return session


def _extract_odata_rows(payload: dict) -> list[dict]:
    '''Extract rows from common OData JSON response shapes.'''
    if 'value' in payload and isinstance(payload['value'], list):
        return payload['value']

    if 'd' in payload:
        d = payload['d']
        if isinstance(d, dict) and 'results' in d and isinstance(d['results'], list):
            return d['results']

    raise ValueError('No OData row collection found in JSON response.')


def inspect_rivm_feed_service(table_id: str) -> None:
    '''Inspect the OData service document for a table.'''
    session = build_session()
    url = f'https://dataderden.cbs.nl/ODataFeed/odata/{table_id}'
    response = session.get(url, timeout=120)
    response.raise_for_status()

    print('Final URL:', response.url)
    print('Status:', response.status_code)
    print('Content-Type:', response.headers.get('Content-Type'))
    print('First 500 characters:')
    print(response.text[:500])


def fetch_rivm_feed_table(table_id: str, entity: str = 'TypedDataSet', top: int = 1000) -> pd.DataFrame:
    '''Fetch rows from an RIVM StatLine OData feed entity and return a dataframe.'''
    session = build_session()
    url = f'https://dataderden.cbs.nl/ODataFeed/odata/{table_id}/{entity}'
    params = {
        '$format': 'json',
        '$top': str(top),
    }

    response = session.get(url, params=params, timeout=120)
    response.raise_for_status()

    content_type = response.headers.get('Content-Type', '').lower()
    text = response.text.lstrip()

    if 'html' in content_type or text.startswith('<!DOCTYPE html') or text.startswith('<html'):
        raise ValueError(
            f'Expected JSON but received HTML from {response.url}. '
            'This suggests the wrong endpoint was used.'
        )

    try:
        payload = response.json()
    except requests.exceptions.JSONDecodeError as exc:
        snippet = response.text[:500]
        raise ValueError(
            f'Could not parse JSON from {response.url}. First 500 characters:\n{snippet}'
        ) from exc

    rows = _extract_odata_rows(payload)
    return pd.DataFrame(rows)


def fetch_rivm_feed_metadata(table_id: str, entity: str) -> pd.DataFrame:
    '''Fetch one metadata entity from the RIVM StatLine feed.'''
    session = build_session()
    url = f'https://dataderden.cbs.nl/ODataFeed/odata/{table_id}/{entity}'
    response = session.get(url, params={'$format': 'json'}, timeout=120)
    response.raise_for_status()

    payload = response.json()
    rows = _extract_odata_rows(payload)
    df = pd.DataFrame(rows)

    output_path = 'data/' + 'RIVM_StatLine_metadata.csv'

    df.to_csv(output_path)

    return df

In [ ]:
inspect_rivm_feed_service('50148NED')

In [ ]:
df = fetch_rivm_feed_table('50148NED', entity='TypedDataSet', top=1000)

output_path = 'data/' + '50148NED_RIVM.csv'

df.to_csv(output_path)


print(df.head())
print(df.shape)
print(df.columns.tolist())

In [ ]:
import re
import pandas as pd
import requests
import xml.etree.ElementTree as ET
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def build_session() -> requests.Session:
    '''Create a requests session with browser-like headers and retry logic.'''
    session = requests.Session()

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=['GET'],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)

    session.headers.update({
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/145.0.0.0 Safari/537.36'
        ),
        'Accept': 'application/json, application/xml, text/plain, */*',
        'Accept-Language': 'en-US,en;q=0.9,nl;q=0.8',
    })
    return session


def _extract_odata_rows(payload: dict) -> list[dict]:
    '''Extract rows from common OData JSON response shapes.'''
    if 'value' in payload and isinstance(payload['value'], list):
        return payload['value']

    if 'd' in payload:
        d = payload['d']
        if isinstance(d, dict) and 'results' in d and isinstance(d['results'], list):
            return d['results']

    raise ValueError('No OData row collection found in JSON response.')


def list_rivm_feed_entities(table_id: str) -> list[str]:
    '''Read the OData service document and return the available entity set names for a table.'''
    session = build_session()
    url = f'https://dataderden.cbs.nl/ODataFeed/odata/{table_id}'
    response = session.get(url, timeout=120)
    response.raise_for_status()

    text = response.text.strip()
    if not text.startswith('<'):
        raise ValueError(f'Expected XML service document from {url}, got:\n{text[:500]}')

    root = ET.fromstring(response.content)

    namespaces = {
        'app': 'http://www.w3.org/2007/app',
        'atom': 'http://www.w3.org/2005/Atom',
    }

    entities: list[str] = []
    for collection in root.findall('.//app:collection', namespaces):
        href = collection.attrib.get('href', '')
        name = href.rstrip('/').split('/')[-1]
        if name:
            entities.append(name)

    if not entities:
        raise ValueError(f'No entity collections found in service document for table {table_id}.')

    return entities


def fetch_rivm_feed_entity(table_id: str, entity: str) -> pd.DataFrame:
    '''Fetch one entity set from the RIVM/CBS OData feed as a dataframe.'''
    session = build_session()
    url = f'https://dataderden.cbs.nl/ODataFeed/odata/{table_id}/{entity}'
    response = session.get(url, params={'$format': 'json'}, timeout=120)
    response.raise_for_status()

    content_type = response.headers.get('Content-Type', '').lower()
    text = response.text.lstrip()

    if 'html' in content_type or text.startswith('<!DOCTYPE html') or text.startswith('<html'):
        raise ValueError(f'Expected JSON but received HTML from {response.url}.')

    if text.startswith('<?xml') or text.startswith('<feed') or text.startswith('<entry'):
        raise ValueError(
            f'Expected JSON but received XML from {response.url}. '
            'This entity may not support JSON at that endpoint.'
        )

    payload = response.json()
    rows = _extract_odata_rows(payload)
    return pd.DataFrame(rows)


def fetch_rivm_statline_dimensions(table_id: str) -> dict[str, pd.DataFrame]:
    '''Fetch all metadata-like entities exposed by an RIVM/CBS OData table.'''
    entities = list_rivm_feed_entities(table_id)

    skip_entities = {'TypedDataSet', 'UntypedDataSet'}
    parts: dict[str, pd.DataFrame] = {}

    for entity in entities:
        if entity in skip_entities:
            continue
        try:
            parts[entity] = fetch_rivm_feed_entity(table_id, entity)
        except Exception as exc:
            print(f'Skipping {entity}: {type(exc).__name__}: {exc}')

    return parts

In [ ]:
entities = list_rivm_feed_entities('50148NED')
print(entities)

meta = fetch_rivm_statline_dimensions('50148NED')
for key, value in meta.items():
    print(f'\n{key}')
    print(value.head())
    print(value.shape) # Each value of meta is a pd DataFrame and should be written to csv on its own.

    output_path = 'data/' + f'50148NED_RIVM_meta_{key}.csv'

    value.to_csv(output_path)

In [ ]:
def fetch_rivm_dimension_entities_only(table_id: str) -> dict[str, pd.DataFrame]:
    '''Fetch only likely dimension tables, excluding dataset rows and general table descriptors.'''
    entities = list_rivm_feed_entities(table_id)

    exclude = {
        'TypedDataSet',
        'UntypedDataSet',
        'TableInfos',
        'DataProperties',
        'CategoryGroups',
    }

    parts: dict[str, pd.DataFrame] = {}
    for entity in entities:
        if entity in exclude:
            continue
        try:
            parts[entity] = fetch_rivm_feed_entity(table_id, entity)
        except Exception as exc:
            print(f'Skipping {entity}: {type(exc).__name__}: {exc}')

    return parts

In [ ]:
dims = fetch_rivm_dimension_entities_only('50148NED')
for key, value in dims.items():
    print(f'\n{key}')
    print(value.head())

    output_path = 'data/' + f'50148NED_RIVM_dims_{key}.csv'

    value.to_csv(output_path)

Status: 200 status code means ok. Need to figure out how to access the data from this request.

Source Name:

RIVM, VZinfo, Acute zorg

Data Description

Curated acute care indicators and contextual information, covering supply and use, includign ambulance services and emergency hospital care context and links to underlying sources. Useful for indicator selection and triangulation across official datasets.

Direct link:

https://www.vzinfo.nl/acute-zorg

Access method:

Web portal with linked datasets and downloadable resources

In [ ]:
%pip install beautifulsoup4

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin


def fetch_vzinfo_acute_zorg_sources() -> pd.DataFrame:
    '''Extract linked source URLs from the VZinfo acute care page.'''
    url = 'https://www.vzinfo.nl/acute-zorg'
    response = requests.get(
        url,
        timeout=60,
        headers={
            'User-Agent': (
                'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                'AppleWebKit/537.36 (KHTML, like Gecko) '
                'Chrome/145.0.0.0 Safari/537.36'
            )
        },
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')
    records = []

    for a in soup.select('a[href]'):
        label = a.get_text(' ', strip=True)
        href = urljoin(url, a['href'])
        if label and ('acute zorg' in label.lower() or 'zorg' in label.lower() or 'bron' in label.lower()):
            records.append({'label': label, 'url': href})

    df = pd.DataFrame(records).drop_duplicates()
    return df.sort_values('label').reset_index(drop=True)


df_sources = fetch_vzinfo_acute_zorg_sources()
print(df_sources.head(20))

In [ ]:
%pip install openpyxl

In [ ]:
import io
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def build_session() -> requests.Session:
    '''Create a resilient session with browser-like headers.'''
    session = requests.Session()

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=[403, 429, 500, 502, 503, 504],
        allowed_methods=['GET'],
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)

    session.headers.update({
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/145.0.0.0 Safari/537.36'
        ),
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9,nl;q=0.8',
    })
    return session


def find_rivm_acute_care_xlsx_url() -> str:
    '''Find the current XLSX download URL from the official RIVM document page.'''
    session = build_session()
    page_url = 'https://www.rivm.nl/documenten/acute-zorg-overzicht-feiten-en-cijfers'

    response = session.get(page_url, timeout=120)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')

    for a in soup.select('a[href]'):
        href = a.get('href', '')
        text = a.get_text(' ', strip=True).lower()

        if '.xlsx' in href.lower():
            return urljoin(page_url, href)

        if 'xlsx' in text:
            return urljoin(page_url, href)

    raise ValueError('Could not find an XLSX link on the RIVM acute care document page.')


def fetch_rivm_acute_care_workbook(
    save_path: str = "data/rivm_acute_care.xlsx"
) -> pd.ExcelFile:
    '''Download the RIVM acute care workbook, save it locally, and return a pandas ExcelFile object.'''

    session = build_session()
    xlsx_url = find_rivm_acute_care_xlsx_url()

    response = session.get(xlsx_url, timeout=120)
    response.raise_for_status()

    # Ensure data directory exists
    path = Path(save_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    # Save file to disk
    with open(path, "wb") as f:
        f.write(response.content)

    # Return ExcelFile object for immediate use
    return pd.ExcelFile(io.BytesIO(response.content))

def preview_sheet_top(workbook: pd.ExcelFile, sheet_name: str, nrows: int = 20) -> pd.DataFrame:
    '''Read the top of a sheet without assuming any header row.'''
    return pd.read_excel(workbook, sheet_name=sheet_name, header=None, nrows=nrows)


wb = fetch_rivm_acute_care_workbook()
print(wb.sheet_names)

for sheet in wb.sheet_names:
    print(f'\n=== {sheet} ===')
    preview = preview_sheet_top(wb, sheet, nrows=15)
    print(preview)

In [ ]:
def read_clean_sheet(
    workbook: pd.ExcelFile,
    sheet_name: str,
    header_row: int,
) -> pd.DataFrame:
    '''Read one sheet using the correct header row and clean empty rows and unnamed columns.'''
    df = pd.read_excel(workbook, sheet_name=sheet_name, header=header_row)

    df = df.dropna(axis=0, how='all')
    df = df.dropna(axis=1, how='all')

    keep_cols = [
        col for col in df.columns
        if not (isinstance(col, str) and col.startswith('Unnamed:'))
    ]
    df = df[keep_cols]

    return df.reset_index(drop=True)

In [ ]:
wb = fetch_rivm_acute_care_workbook()

df = read_clean_sheet(
    workbook=wb,
    sheet_name=wb.sheet_names[0],
    header_row=6,
)

print(df.head())
print(df.shape)
print(df.columns.tolist())

output_path = "data/rivm_acute_care.csv"

df.to_csv(output_path)

Status: Code 200 means OK. Site reached, and data is accessible.

Source name:

Kadaster, PDOK Bestuurlijke Gebieden, OGC services and APIs

Data description:

Official administrative areas including municipalities, provinces, and national border, supporting consistent geographic linkage and aggregation of health indicators. Available via standard OGC services and modern OGC APIs

Direct link:

https://www.pdok.nl/introductie/-/article/bestuurlijke-gebieden

Access method:

WFS and OGC API Features, plus downloads

In [ ]:
%pip install geopandas

In [ ]:
import geopandas as gpd
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def build_session() -> requests.Session:
    '''Create a resilient requests session for PDOK API calls.'''
    session = requests.Session()

    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=['GET'],
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)

    session.headers.update({
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/145.0.0.0 Safari/537.36'
        ),
        'Accept': 'application/geo+json,application/json,*/*',
        'Accept-Language': 'en-US,en;q=0.9,nl;q=0.8',
    })
    return session


def list_pdok_bg_collections() -> pd.DataFrame:
    '''List available PDOK Bestuurlijke Gebieden collections.'''
    session = build_session()
    url = 'https://api.pdok.nl/kadaster/bestuurlijkegebieden/ogc/v1/collections'
    response = session.get(url, params={'f': 'json'}, timeout=60)
    response.raise_for_status()

    payload = response.json()
    rows: list[dict[str, str | None]] = []

    for item in payload.get('collections', []):
        rows.append({
            'id': item.get('id'),
            'title': item.get('title'),
            'description': item.get('description'),
        })

    return pd.DataFrame(rows)


def find_municipality_collection_id() -> str:
    '''Return the municipality collection id.'''
    collections = list_pdok_bg_collections()

    mask = (
        collections['id'].fillna('').str.contains('gemeente', case=False, regex=False) |
        collections['title'].fillna('').str.contains('gemeente', case=False, regex=False) |
        collections['description'].fillna('').str.contains('gemeente', case=False, regex=False)
    )

    matches = collections.loc[mask]
    if matches.empty:
        raise ValueError(
            'No municipality collection found. Inspect list_pdok_bg_collections() output.'
        )

    return str(matches.iloc[0]['id'])


def fetch_pdok_collection_items(
    collection_id: str,
    limit: int = 100,
    crs: str = 'http://www.opengis.net/def/crs/OGC/1.3/CRS84',
) -> gpd.GeoDataFrame:
    '''Fetch items from one PDOK Bestuurlijke Gebieden collection as a GeoDataFrame.'''
    session = build_session()
    url = f'https://api.pdok.nl/kadaster/bestuurlijkegebieden/ogc/v1/collections/{collection_id}/items'
    params = {
        'f': 'json',
        'limit': limit,
        'crs': crs,
    }

    response = session.get(url, params=params, timeout=120)
    response.raise_for_status()

    payload = response.json()
    return gpd.GeoDataFrame.from_features(payload['features'], crs='EPSG:4326')


def fetch_pdok_municipalities(limit: int = 100) -> gpd.GeoDataFrame:
    '''Fetch municipality polygons from PDOK Bestuurlijke Gebieden.'''
    collection_id = find_municipality_collection_id()
    return fetch_pdok_collection_items(collection_id=collection_id, limit=limit)


collections_df = list_pdok_bg_collections()
print(collections_df)

gdf = fetch_pdok_municipalities(limit=50)
print(gdf.head())
print(gdf.columns.tolist())
print(gdf.shape)

output_path = Path("data/raw/pdok_municipalities_limit50.gpkg")
output_path.parent.mkdir(parents=True, exist_ok=True)

gdf.to_file(output_path, layer="municipalities", driver="GPKG")

Status: 400. Cannot or will not fulfil request for something that is expected to be a client error.

Source name:

Kadaster adn CBS, PDOK CBS Wijken en Buurten (geometries with key figures)

Data description:

Neighbourhood and district geometries with selected CBS key figures, enabling small area analyssi, mapping, and linkage for deprivation and health gradients. Suitable as a spatial denominator layer for localized health system studies.

Direct Link:

https://www.pdok.nl/ogc-webservices/-/article/cbs-wijken-en-buurten

Access method:

WFS and WMS services

In [ ]:
import geopandas as gpd
import requests

def read_wijken_buurten_wfs(max_features: int = 200) -> gpd.GeoDataFrame:
    '''Read a sample from the CBS Wijken en Buurten WFS as GeoJSON.'''
    base = 'https://service.pdok.nl/cbs/wijkenbuurten/2023/wfs/v1_0'
    params = {
        'service': 'WFS',
        'request': 'GetFeature',
        'version': '2.0.0',
        'typeNames': 'wijkenbuurten:buurten',
        'outputFormat': 'application/json',
        'count': str(max_features),
    }
    r = requests.get(base, params=params, timeout=60)
    r.raise_for_status()
    geojson = r.json()
    return gpd.GeoDataFrame.from_features(geojson['features'], crs='EPSG:4326')

gdf = read_wijken_buurten_wfs(max_features=50)
print(gdf.head())

output_path = Path("data/raw/pdok_neighborhoods_maxfeatures50.gpkg")
output_path.parent.mkdir(parents=True, exist_ok=True)

gdf.to_file(output_path, layer="municipalities", driver="GPKG")